# WSP on Colab

Select a hosted GPU runtime. The download cell prepares the public Sketchy v1 dataset in the current VM.

In [ ]:
from pathlib import Path
import subprocess

CONTENT = Path("/content")
if not CONTENT.is_dir():
    raise RuntimeError("Select a hosted Colab runtime.")
REPO = CONTENT / "WSP"
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/HieuNM1804/WSP.git", str(REPO)], check=True)
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], check=True)

In [ ]:
import sys

SKETCHY = CONTENT / "Sketchy"
subprocess.run([sys.executable, str(REPO / "wsp_sbir/download_sketchy.py"), "--destination", str(SKETCHY)], check=True)

## Prepare WSP

Setup checks the sandbox, reuses Colab’s GPU PyTorch, and installs a CPU environment for sandboxed Python.

In [ ]:
import os

subprocess.run(["bash", str(REPO / "wsp_sbir/setup_colab.sh"), str(SKETCHY)], check=True)
CODEX = REPO / ".colab-tools/node_modules/.bin/codex"

## Run WSP

Sign in to Codex on this runtime before starting a run. The optional login cell requests a device code. Run the next cell with a budget of 2 to smoke-test the complete workflow, or set your intended budget.

In [ ]:
login = False
if login:
    subprocess.run([str(CODEX), "login", "--device-auth"], check=True)
else:
    subprocess.run([str(CODEX), "login", "status"], check=False)

In [ ]:
budget = 2
name = f"s2_astra_{budget}"
run = REPO / "wsp_sbir/runs" / name
launcher_env = dict(os.environ, WSP_CODEX_BIN=str(CODEX), WSP_SANDBOX_VENV=str(REPO / "wsp_sbir/.venv_sandbox"))
subprocess.run([str(CODEX), "login", "status"], check=True)
if (run / "state.json").exists():
    args = ["--run", str(run), "--resume"]
else:
    args = ["--task", "tasks/sketchy_2", "--run", str(run), "--budget", str(budget)]
subprocess.run(["bash", str(REPO / "wsp_sbir/run_local.sh"), *args], check=True, env=launcher_env)

## Publish a result

Set the run directory below. The cell previews the metrics JSON. Once Git is authenticated with write access to the repository, set `publish = True` to push a `results/<run>` branch.

In [ ]:
publish = False
command = [sys.executable, str(REPO / "wsp_sbir/publish_results.py"), "--run", str(run)]
if publish:
    command.append("--push")
subprocess.run(command, cwd=REPO, check=True)